# Notebook 1 . Join DB

Gabungkan beberapa file CSV berdasarkan kolom ID, lalu simpan hasilnya sebagai CSV. Atur nama/path file, kolom ID, dan jenis join di sel konfigurasi. Kolom dibaca sebagai teks supaya ID seperti `00123` tetap utuh.

In [ ]:
from pathlib import Path
import pandas as pd

# PENGATURAN YANG BISA DIUBAH
Input_CSV = "Input/tabel_utama.csv"
Kolom_ID_Utama = "id"
Separator_CSV = ","  # Ganti menjadi ";" jika CSV memakai titik koma

Daftar_CSV_Tambahan = [
    # Isi nama, file, kolom_id, dan jenis_gabung untuk tiap CSV tambahan.
    # jenis_gabung dapat berupa left, inner, right, atau outer.
    {"nama": "tambahan", "file": "Input/tabel_tambahan.csv", "kolom_id": "id", "jenis_gabung": "left"},
    # {"nama": "referensi", "file": "Input/referensi.csv", "kolom_id": "kode", "jenis_gabung": "inner"},
]

Output_CSV = Path("Output/hasil_join.csv")
Encoding_Output = "utf-8-sig"
Rapikan_Spasi = True
Hapus_ID_Duplikat = False  # Simpan baris pertama jika ID berulang di setiap file

def find_project_dir():
    current = Path.cwd().resolve()
    for folder in (current, *current.parents):
        if (folder / "Input").is_dir():
            (folder / "Output").mkdir(parents=True, exist_ok=True)
            return folder
    return current

BASE_DIR = find_project_dir()

TABLES = [
    {"nama": "utama", "file": Input_CSV, "kolom_id": Kolom_ID_Utama},
    *Daftar_CSV_Tambahan,
]


## Baca file CSV

Periksa ringkasan jumlah baris, kolom, ID kosong, dan ID duplikat sebelum join.

In [ ]:
def load_table(config):
    path = Path(config["file"])
    if not path.is_absolute():
        path = BASE_DIR / path
    if not path.exists():
        raise FileNotFoundError(f"File '{config['nama']}' tidak ditemukan: {path}")

    df = pd.read_csv(path, sep=Separator_CSV, dtype="string", encoding="utf-8-sig")
    df.columns = df.columns.astype(str).str.strip()
    id_col = config["kolom_id"]
    if id_col not in df.columns:
        raise KeyError(f"Kolom ID '{id_col}' tidak ada di {path.name}. Kolom tersedia: {df.columns.tolist()}")

    if Rapikan_Spasi:
        for col in df.select_dtypes(include=["string", "object"]).columns:
            df[col] = df[col].str.strip()
    df[id_col] = df[id_col].str.strip()

    missing_ids = int(df[id_col].isna().sum() + df[id_col].eq("").sum())
    duplicate_ids = int(df.loc[df[id_col].notna() & df[id_col].ne(""), id_col].duplicated().sum())
    print(f"{config['nama']}: {len(df):,} baris, {len(df.columns)} kolom, {missing_ids:,} ID kosong, {duplicate_ids:,} ID duplikat")
    if Hapus_ID_Duplikat:
        before = len(df)
        df = df.drop_duplicates(subset=[id_col], keep="first")
        print(f"  Dihapus karena ID duplikat: {before - len(df):,} baris")
    return df

if not TABLES:
    raise ValueError("Konfigurasikan minimal satu file CSV.")
tables = {config["nama"]: load_table(config) for config in TABLES}
display(pd.DataFrame([{"tabel": name, "baris": len(df), "kolom": len(df.columns)} for name, df in tables.items()]))


## Join berdasarkan ID

Pilih `left`, `inner`, `right`, atau `outer` pada konfigurasi setiap file tambahan. Nama kolom ID boleh berbeda. ID kosong di file tambahan tidak dianggap cocok. Jika ada nama kolom selain ID yang sama, kolom tambahan diberi akhiran nama tabel.

In [ ]:
main_config = TABLES[0]
result = tables[main_config["nama"]].copy()
main_id = main_config["kolom_id"]

for config in TABLES[1:]:
    right = tables[config["nama"]].copy()
    right_id = config["kolom_id"]
    how = config.get("jenis_gabung", "left")
    if how not in {"left", "inner", "right", "outer"}:
        raise ValueError(f"Jenis join tidak dikenal untuk '{config['nama']}': {how}")

    right = right.loc[right[right_id].notna() & right[right_id].ne("")]
    overlap = (set(result.columns) & set(right.columns)) - {main_id, right_id}
    suffixes = ("", f"_{config['nama']}")
    if right_id != main_id:
        right = right.rename(columns={right_id: main_id})
    result = result.merge(right, on=main_id, how=how, suffixes=suffixes, sort=False)
    print(f"Join {config['nama']} ({how}) via '{main_id}': {len(result):,} baris; kolom bertabrakan: {sorted(overlap)}")

display(result.head())
print(f"Ukuran hasil: {result.shape[0]:,} baris x {result.shape[1]} kolom")


## Simpan hasil join

In [ ]:
output_path = Output_CSV if Output_CSV.is_absolute() else BASE_DIR / Output_CSV
output_path.parent.mkdir(parents=True, exist_ok=True)
result.to_csv(output_path, index=False, encoding=Encoding_Output, sep=Separator_CSV)
print(f"Tersimpan: {output_path.resolve()}")